#### Create Vector Database using ChromaDB

In [1]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
import chromadb
import os
import json
import re
from datetime import datetime
from collections import defaultdict
from rank_bm25 import BM25Okapi

from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "")


#### Create Vector Database Client

In [ ]:
# Initialize the Chroma Cloud client
CHROMA_API_KEY = os.getenv("CHROMA_API_KEY")
CHROMA_TENANT = (os.getenv("CHROMA_TENANT_ID"))
CHROMA_DATABASE = (os.getenv("CHROMA_DATABASE_ID"))

missing_chroma_env = [
    name
    for name, value in {
        "CHROMA_API_KEY": CHROMA_API_KEY,
        "CHROMA_TENANT": CHROMA_TENANT,
        "CHROMA_DATABASE": CHROMA_DATABASE,
    }.items()
    if not value
]

if missing_chroma_env:
    raise ValueError(
        "Missing Chroma Cloud environment variable(s): "
        + ", ".join(missing_chroma_env)
    )

raw_client = chromadb.CloudClient(
    api_key=CHROMA_API_KEY,
    tenant=CHROMA_TENANT,
    database=CHROMA_DATABASE,
)

# Resume collection
resume_collection = Chroma(
    client=raw_client,
    collection_name="resumes",
    collection_metadata={"hnsw:space": "cosine"}
)

# JD collection
jd_collection = Chroma(
    client=raw_client,
    collection_name="jds",
    collection_metadata={"hnsw:space": "cosine"}
)

print("Chroma Cloud collections created/loaded successfully")


Chroma Cloud collections created/loaded successfully


#### Verify Chroma Cloud Connection


In [5]:
cloud_collection_counts = {
    collection.name: collection.count()
    for collection in raw_client.list_collections()
}

print("Connected to Chroma Cloud")
print(f"Collections visible: {list(cloud_collection_counts.keys())}")
print(f"Collection counts: {cloud_collection_counts}")


Connected to Chroma Cloud
Collections visible: ['resumes', 'jds']
Collection counts: {'resumes': 0, 'jds': 0}


#### Load Embeddings

In [6]:
RESUME_EMBEDDINGS_PATH = r"..\embeddings\all_resume_embeddings.json"
JD_EMBEDDINGS_PATH = r"..\embeddings\all_jd_embeddings.json"

with open(RESUME_EMBEDDINGS_PATH, "r", encoding="utf-8") as f:
    resume_embeddings = json.load(f)

with open(JD_EMBEDDINGS_PATH, "r", encoding="utf-8") as f:
    jd_embeddings = json.load(f)

print(f"Loaded resume embeddings for {len(resume_embeddings)} resume file(s).")
print(f"Loaded JD embeddings for {len(jd_embeddings)} JD file(s).")


Loaded resume embeddings for 7 resume file(s).
Loaded JD embeddings for 3 JD file(s).


#### Upload Embeddings to Chroma Cloud


In [7]:
def get_stored_files(collection: Chroma) -> set:
    """
    Returns filenames already stored in Chroma Cloud.
    Duplicate checking is based on metadata['source_file'].
    """

    results = collection._collection.get(include=["metadatas"])
    metadatas = results.get("metadatas") or []

    return {
        metadata.get("source_file")
        for metadata in metadatas
        if metadata and metadata.get("source_file")
    }


def filter_new_embeddings(embeddings_dict, collection: Chroma):
    """
    Keeps only embeddings whose filename is not already present in Chroma Cloud.
    """

    already_stored = get_stored_files(collection)

    new_embeddings = {}
    skipped_files = []

    for filename, chunks in embeddings_dict.items():
        if filename in already_stored:
            skipped_files.append(filename)
        else:
            new_embeddings[filename] = chunks

    return new_embeddings, skipped_files


def store_embeddings(embeddings_dict, collection: Chroma, label="chunks", batch_size=100):
    """
    Uploads only new files to Chroma Cloud.
    Old resumes/JDs are skipped based on source_file metadata.
    """

    new_embeddings, skipped_files = filter_new_embeddings(
        embeddings_dict,
        collection
    )

    ids = []
    embeddings = []
    documents = []
    metadatas = []

    for filename, chunks in new_embeddings.items():
        for i, chunk in enumerate(chunks):
            ids.append(f"{filename}_chunk_{i}")
            embeddings.append(chunk["embedding"])
            documents.append(chunk["content"])
            metadatas.append({
                "source_file": filename,
                "type": chunk["type"],
                "chunk_index": i
            })

    if skipped_files:
        print(f"Skipped {len(skipped_files)} already stored {label} file(s).")

    if not ids:
        print(f"No new {label} to upload. All files are already in Chroma Cloud.")
        return {
            "stored_files": list(new_embeddings.keys()),
            "skipped_files": skipped_files,
            "stored_chunks": 0,
            "total_chunks": collection._collection.count()
        }

    for start in range(0, len(ids), batch_size):
        end = start + batch_size
        collection._collection.add(
            ids=ids[start:end],
            embeddings=embeddings[start:end],
            documents=documents[start:end],
            metadatas=metadatas[start:end]
        )

    print(f"Uploaded {len(ids)} new {label} to Chroma Cloud.")
    print(f"New file(s) added: {list(new_embeddings.keys())}")

    return {
        "stored_files": list(new_embeddings.keys()),
        "skipped_files": skipped_files,
        "stored_chunks": len(ids),
        "total_chunks": collection._collection.count()
    }


In [8]:
resume_store_result = store_embeddings(
    resume_embeddings,
    resume_collection,
    label="resume chunks"
)

jd_store_result = store_embeddings(
    jd_embeddings,
    jd_collection,
    label="JD chunks"
)

print("\nChroma Cloud Storage Summary")
print(f"Resume files added: {len(resume_store_result['stored_files'])}")
print(f"Resume files skipped: {len(resume_store_result['skipped_files'])}")
print(f"Resume chunks added: {resume_store_result['stored_chunks']}")
print(f"Resume chunks in cloud: {resume_store_result['total_chunks']}")

print(f"JD files added: {len(jd_store_result['stored_files'])}")
print(f"JD files skipped: {len(jd_store_result['skipped_files'])}")
print(f"JD chunks added: {jd_store_result['stored_chunks']}")
print(f"JD chunks in cloud: {jd_store_result['total_chunks']}")


Uploaded 53 new resume chunks to Chroma Cloud.
New file(s) added: ['Abhishek_Shaurya_Resume.pdf', 'Jay Kumar Behera_CV_DS.pdf', 'Komal_Kamble_1 (1).pdf', 'MeetLad_Resume.pdf', 'Prity-Kumari-Resume-DevOps-2025.pdf', 'SOUMYADEEP_SEN_CV.pdf', 'MananDaxini-Updated-2026-t (1) (2).pdf']
Uploaded 17 new JD chunks to Chroma Cloud.
New file(s) added: ['DS_JD.txt', 'PM_JD.txt', 'SWE_JD.txt']

Chroma Cloud Storage Summary
Resume files added: 7
Resume files skipped: 0
Resume chunks added: 53
Resume chunks in cloud: 53
JD files added: 3
JD files skipped: 0
JD chunks added: 17
JD chunks in cloud: 17


Final Method

In [9]:
# Hybrid search weights
VECTOR_CHUNK_WEIGHT = 0.80
BM25_CHUNK_WEIGHT = 0.20

# Final candidate score weights
RELEVANCE_WEIGHT = 0.70
ROLE_FIT_WEIGHT = 0.20
EXPERIENCE_FIT_WEIGHT = 0.10

JD_CHUNK_WEIGHTS = {
    "skills": 0.35,
    "responsibilities": 0.30,
    "qualifications": 0.20,
    "overview": 0.10,
    "summary": 0.10,
    "experience": 0.10,
    "education": 0.05
}

ALL_RESUME_CHUNKS_PATH = r"..\chunks\all_resume_chunks.json"
ALL_JD_PATH = r"..\output_json_jd\all_jd.json"
ALL_RESUME_PATH = r"..\output_jsons_resume\all_resumes.json"


def tokenize_for_bm25(text):
    if not text:
        return []
    return re.findall(r"[a-zA-Z0-9+#.]+", text.lower())


def load_resume_chunks_for_bm25(chunks_path=ALL_RESUME_CHUNKS_PATH):
    with open(chunks_path, "r", encoding="utf-8") as f:
        all_resume_chunks = json.load(f)

    documents = []
    for filename, chunks in all_resume_chunks.items():
        for i, chunk in enumerate(chunks):
            documents.append({
                "source_file": filename,
                "chunk_type": chunk.get("type"),
                "chunk_index": i,
                "content": chunk.get("content", ""),
            })

    tokenized_corpus = [tokenize_for_bm25(doc["content"]) for doc in documents]
    bm25_index = BM25Okapi(tokenized_corpus)

    print(f"BM25 index built with {len(documents)} resume chunks.")
    return bm25_index, documents


def normalize_bm25_scores(raw_scores):
    if raw_scores is None:
        return []

    raw_scores = list(raw_scores)
    if not raw_scores:
        return []

    min_score = min(raw_scores)
    max_score = max(raw_scores)

    if max_score == min_score:
        return [0.0 for _ in raw_scores]

    return [
        (score - min_score) / (max_score - min_score)
        for score in raw_scores
    ]


def search_bm25_resume_chunks(jd_chunk, bm25_index, bm25_documents):
    query_tokens = tokenize_for_bm25(jd_chunk["content"])
    raw_scores = bm25_index.get_scores(query_tokens)
    normalized_scores = normalize_bm25_scores(raw_scores)

    bm25_results = {}
    for doc, raw_score, norm_score in zip(bm25_documents, raw_scores, normalized_scores):
        key = (doc["source_file"], doc["chunk_index"])
        bm25_results[key] = {
            "bm25_score": round(float(raw_score), 4),
            "bm25_score_norm": round(float(norm_score), 4),
        }

    return bm25_results


def get_jd_chunks(jd_filename, jd_collection):
    results = jd_collection._collection.get(
        where={"source_file": jd_filename},
        include=["embeddings", "documents", "metadatas"]
    )

    if not results["ids"]:
        print(f"No chunks found for JD: {jd_filename}")
        return []

    chunks = []
    for i in range(len(results["ids"])):
        chunk_type = results["metadatas"][i]["type"]
        chunks.append({
            "jd_chunk_id": f"{jd_filename}__{chunk_type}__{i}",
            "type": chunk_type,
            "content": results["documents"][i],
            "embedding": results["embeddings"][i]
        })

    print(f"Fetched {len(chunks)} chunks for JD: {jd_filename}")
    return chunks


def search_similar_resumes(jd_chunk, resume_collection, top_k,
                           bm25_index=None, bm25_documents=None):
    results = resume_collection._collection.query(
        query_embeddings=[jd_chunk["embedding"]],
        n_results=top_k,
        include=["metadatas", "distances"]
    )

    bm25_results = {}
    if bm25_index is not None and bm25_documents is not None:
        bm25_results = search_bm25_resume_chunks(
            jd_chunk,
            bm25_index,
            bm25_documents
        )

    matches = []

    for i in range(len(results["ids"][0])):
        metadata = results["metadatas"][0][i]
        distance = results["distances"][0][i]
        vector_similarity = round(1 - distance, 4)

        source_file = metadata["source_file"]
        chunk_index = metadata.get("chunk_index")

        bm25_match = bm25_results.get((source_file, chunk_index), {})
        bm25_score = bm25_match.get("bm25_score", 0.0)
        bm25_score_norm = bm25_match.get("bm25_score_norm", 0.0)

        hybrid_score = round(
            (VECTOR_CHUNK_WEIGHT * vector_similarity) +
            (BM25_CHUNK_WEIGHT * bm25_score_norm),
            4
        )

        matches.append({
            "source_file": source_file,
            "chunk_type": metadata["type"],
            "chunk_index": chunk_index,
            "similarity": vector_similarity,
            "bm25_score": bm25_score,
            "bm25_score_norm": bm25_score_norm,
            "hybrid_score": hybrid_score,
            "jd_chunk_id": jd_chunk["jd_chunk_id"],
            "jd_chunk_type": jd_chunk["type"]
        })

    return matches


def aggregate_scores(all_matches):
    candidate_jd_scores = defaultdict(dict)

    for match in all_matches:
        filename = match["source_file"]
        jd_chunk_id = match["jd_chunk_id"]
        hybrid_score = match["hybrid_score"]

        current_best = candidate_jd_scores[filename].get(jd_chunk_id)

        if current_best is None or hybrid_score > current_best["hybrid_score"]:
            candidate_jd_scores[filename][jd_chunk_id] = {
                "hybrid_score": hybrid_score,
                "similarity": match["similarity"],
                "bm25_score": match.get("bm25_score", 0.0),
                "bm25_score_norm": match.get("bm25_score_norm", 0.0),
                "resume_chunk_type": match["chunk_type"],
                "jd_chunk_type": match["jd_chunk_type"]
            }

    relevance_scores = {}

    for filename, jd_scores in candidate_jd_scores.items():
        weighted_sum = 0.0
        total_weight = 0.0
        chunk_scores = []

        for jd_chunk_id, v in jd_scores.items():
            jd_type = v["jd_chunk_type"]
            hybrid_score = v["hybrid_score"]
            weight = JD_CHUNK_WEIGHTS.get(jd_type, 0.05)

            weighted_sum += hybrid_score * weight
            total_weight += weight

            chunk_scores.append({
                "jd_chunk_id": jd_chunk_id,
                "jd_chunk_type": jd_type,
                "resume_chunk_type": v["resume_chunk_type"],
                "similarity": v["similarity"],
                "bm25_score": v["bm25_score"],
                "bm25_score_norm": v["bm25_score_norm"],
                "hybrid_score": hybrid_score,
                "weight": weight,
                "contribution": round(hybrid_score * weight, 4)
            })

        final_score = round(
            weighted_sum / total_weight,
            4
        ) if total_weight > 0 else 0.0

        relevance_scores[filename] = {
            "final_score": final_score,
            "chunk_scores": sorted(
                chunk_scores,
                key=lambda x: x["hybrid_score"],
                reverse=True
            )
        }

    return relevance_scores


def parse_date(date_str):
    if not date_str or "present" in date_str.lower():
        return datetime.today()

    for fmt in ("%m/%Y", "%b %Y", "%B %Y", "%Y"):
        try:
            return datetime.strptime(date_str.strip(), fmt)
        except ValueError:
            pass

    return None


def compute_experience_years(experience_list):
    total_months = 0

    for exp in (experience_list or []):
        start = parse_date(exp.get("start_date"))
        end = parse_date(exp.get("end_date"))

        if not start or not end:
            continue

        total_months += max(
            0,
            (end.year - start.year) * 12 + (end.month - start.month)
        )

    return round(total_months / 12, 1)


def extract_jd_min_years(jd_parsed):
    exp_str = jd_parsed.get("experience_required") or ""
    match = re.search(r"(\d+)", exp_str)
    return float(match.group(1)) if match else 0.0


def get_text(value):
    if isinstance(value, list):
        return " ".join(get_text(v) for v in value)
    if isinstance(value, dict):
        return " ".join(get_text(v) for v in value.values())
    return str(value or "")


def compute_role_fit_score(jd_parsed, resume_parsed):
    jd_text = get_text(jd_parsed).lower()
    resume_text = get_text(resume_parsed).lower()

    if "software" in jd_text or "backend" in jd_text or "frontend" in jd_text:
        role_groups = {
            "software_title": [
                "software engineer", "software developer", "backend developer",
                "full stack", "full-stack", "developer", "system engineer"
            ],
            "backend": [
                "python", "java", "node", "spring boot", ".net", "c#"
            ],
            "api_microservices": [
                "api", "rest", "microservice", "microservices", "web api"
            ],
            "frontend": [
                "react", "angular", "javascript", "typescript"
            ],
            "database": [
                "sql", "mysql", "postgresql", "mongodb", "dynamodb",
                "nosql", "relational database"
            ],
            "cloud_devops": [
                "aws", "ci/cd", "jenkins", "docker", "kubernetes",
                "gitlab", "github actions"
            ],
            "ai_systems": [
                "llm", "genai", "agent", "agents", "mcp",
                "langchain", "bedrock", "vector db"
            ]
        }

    elif "data scientist" in jd_text or "machine learning" in jd_text:
        role_groups = {
            "data_title": [
                "data scientist", "machine learning", "ml engineer",
                "data analyst"
            ],
            "programming": [
                "python", "sql", "pyspark"
            ],
            "ml": [
                "machine learning", "predictive", "regression",
                "classification"
            ],
            "analytics": [
                "eda", "analysis", "visualization", "tableau"
            ],
            "statistics": [
                "a/b testing", "hypothesis", "statistics"
            ],
            "ai": [
                "llm", "genai", "langchain", "agents"
            ]
        }

    elif "product" in jd_text:
        role_groups = {
            "product_title": [
                "product owner", "product manager", "business analyst"
            ],
            "product": [
                "roadmap", "backlog", "prd", "brd", "mvp"
            ],
            "stakeholder": [
                "stakeholder", "cross-functional", "scrum", "agile"
            ],
            "analytics": [
                "sql", "dashboard", "metrics", "experimentation"
            ],
            "ai_product": [
                "ai", "llm", "genai"
            ]
        }

    else:
        return 0.5, ["generic_role"]

    matched_groups = []

    for group_name, keywords in role_groups.items():
        if any(keyword in resume_text for keyword in keywords):
            matched_groups.append(group_name)

    role_fit_score = len(matched_groups) / len(role_groups)

    return round(role_fit_score, 4), matched_groups


def compute_experience_fit_score(resume_years, jd_min_years):
    if jd_min_years == 0:
        return 1.0

    return round(min(resume_years / jd_min_years, 1.0), 4)


def get_top_candidates(jd_filename, resume_collection,
                       jd_collection, top_n=3, top_k=60):
    jd_chunks = get_jd_chunks(jd_filename, jd_collection)

    if not jd_chunks:
        return []

    bm25_index, bm25_documents = load_resume_chunks_for_bm25()

    all_matches = []

    for jd_chunk in jd_chunks:
        matches = search_similar_resumes(
            jd_chunk=jd_chunk,
            resume_collection=resume_collection,
            top_k=top_k,
            bm25_index=bm25_index,
            bm25_documents=bm25_documents
        )
        all_matches.extend(matches)

    print(f"Total matches: {len(all_matches)} ({len(jd_chunks)} JD chunks x top_k={top_k})")

    relevance_scores = aggregate_scores(all_matches)

    with open(ALL_JD_PATH, "r", encoding="utf-8") as f:
        all_jds = json.load(f)

    with open(ALL_RESUME_PATH, "r", encoding="utf-8") as f:
        all_resumes = json.load(f)

    jd_parsed = all_jds.get(jd_filename, {})

    if not jd_parsed:
        print(f"JD not found in all_jd.json: {jd_filename}")
        return []

    jd_min_years = extract_jd_min_years(jd_parsed)
    print(f"JD requires minimum {jd_min_years} years experience")

    combined_scores = {}

    for filename, score_data in relevance_scores.items():
        resume_parsed = all_resumes.get(filename, {})

        if not resume_parsed:
            print(f"Resume not found: {filename}")
            continue

        relevance_score = score_data["final_score"]

        resume_years = compute_experience_years(
            resume_parsed.get("experience", [])
        )

        role_fit_score, matched_role_groups = compute_role_fit_score(
            jd_parsed,
            resume_parsed
        )

        experience_fit_score = compute_experience_fit_score(
            resume_years,
            jd_min_years
        )

        final_score = round(
            (RELEVANCE_WEIGHT * relevance_score) +
            (ROLE_FIT_WEIGHT * role_fit_score) +
            (EXPERIENCE_FIT_WEIGHT * experience_fit_score),
            4
        )

        combined_scores[filename] = {
            "final_score": final_score,
            "percentage": f"{round(final_score * 100, 2)}%",
            "relevance_score": f"{round(relevance_score * 100, 2)}%",
            "role_fit_score": f"{round(role_fit_score * 100, 2)}%",
            "experience_fit_score": f"{round(experience_fit_score * 100, 2)}%",
            "resume_years": resume_years,
            "required_years": jd_min_years,
            "matched_role_groups": matched_role_groups,
            "chunk_scores": score_data["chunk_scores"]
        }

    ranked = sorted(
        combined_scores.items(),
        key=lambda x: x[1]["final_score"],
        reverse=True
    )

    print(f"\nTop {top_n} Candidates for JD: {jd_filename}")
    print("Score = 70% relevance + 20% role fit + 10% experience fit")
    print(f"Hybrid chunk score = {int(VECTOR_CHUNK_WEIGHT * 100)}% vector + {int(BM25_CHUNK_WEIGHT * 100)}% BM25")
    print("=" * 75)

    for rank, (filename, scores) in enumerate(ranked[:top_n], start=1):
        print(f"\n#{rank} - {filename}")
        print(f"     Final Score     : {scores['percentage']}")
        print(f"     Relevance Score : {scores['relevance_score']}")
        print(f"     Role Fit        : {scores['role_fit_score']}")
        print(f"     Experience Fit  : {scores['experience_fit_score']}")
        print(f"     Experience      : {scores['resume_years']} yrs")
        print(f"     Matched Groups  : {', '.join(scores['matched_role_groups'])}")

        print("\n     JD Requirement Breakdown (JD-Centric):")
        print(f"       {'JD Requirement':<20} {'Resume Section':<20} {'Vector':>8} {'BM25':>8} {'Hybrid':>8}")
        print(f"       {'-' * 70}")

        for chunk in scores["chunk_scores"]:
            print(f"       {chunk['jd_chunk_type']:<20} "
                  f"{chunk['resume_chunk_type']:<20} "
                  f"{chunk['similarity']:>8.4f} "
                  f"{chunk['bm25_score_norm']:>8.4f} "
                  f"{chunk['hybrid_score']:>8.4f}")

    print(f"\nFull Ranking (all {len(ranked)} candidates):")
    print("-" * 80)

    for rank, (filename, scores) in enumerate(ranked, start=1):
        bar_len = int(scores["final_score"] * 30)
        bar = "#" * bar_len + "-" * (30 - bar_len)

        print(f"  #{rank:<3} {filename:<40} {scores['percentage']:>7}  "
              f"[rel: {scores['relevance_score']} | "
              f"role: {scores['role_fit_score']} | "
              f"exp: {scores['experience_fit_score']}]  {bar}")

    return ranked[:top_n]

In [10]:
# Chroma Cloud vector search smoke test
with open(r"..\output_json_jd\all_jd.json", "r", encoding="utf-8") as f:
    all_jds = json.load(f)

with open(r"..\output_jsons_resume\all_resumes.json", "r", encoding="utf-8") as f:
    all_resumes = json.load(f)

smoke_test_jd = "DS_JD.txt"
top_k = resume_collection._collection.count()

cloud_top_candidates = get_top_candidates(
    jd_filename=smoke_test_jd,
    resume_collection=resume_collection,
    jd_collection=jd_collection,
    top_n=3,
    top_k=top_k
)

print("\nChroma Cloud vector search smoke test")
print(f"JD: {smoke_test_jd}")
print(f"Resume chunks searched: {top_k}")
print("Top candidates:")
for rank, (filename, scores) in enumerate(cloud_top_candidates, start=1):
    print(f"{rank}. {filename} - {scores['percentage']}")


Fetched 5 chunks for JD: DS_JD.txt
BM25 index built with 53 resume chunks.
Total matches: 265 (5 JD chunks x top_k=53)
JD requires minimum 3.0 years experience

Top 3 Candidates for JD: DS_JD.txt
Score = 70% relevance + 20% role fit + 10% experience fit
Hybrid chunk score = 80% vector + 20% BM25

#1 - Komal_Kamble_1 (1).pdf
     Final Score     : 78.17%
     Relevance Score : 69.76%
     Role Fit        : 100.0%
     Experience Fit  : 93.33%
     Experience      : 2.8 yrs
     Matched Groups  : data_title, programming, ml, analytics, statistics, ai

     JD Requirement Breakdown (JD-Centric):
       JD Requirement       Resume Section         Vector     BM25   Hybrid
       ----------------------------------------------------------------------
       skills               skills                 0.7159   0.7042   0.7136
       qualifications       summary                0.6560   0.9421   0.7132
       summary              experience             0.6511   0.8928   0.6994
       responsibil

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm_gemini = ChatGoogleGenerativeAI(
    model        = "gemini-2.5-flash",
    temperature  = 0,
    max_tokens   = 4096
)
from langchain_groq import ChatGroq

llm_groq = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

#### Generate the Summary with Template

In [17]:
def clean_json_str(json_str):
    # Remove trailing commas before } or ]
    json_str = re.sub(r',\s*([}\]])', r'\1', json_str)
    return json_str

def generate_candidate_summary(candidate_filename, jd_filename,
                                all_resumes, all_jds, llm,
                                scores):
    """
    Generates a structured summary for a single candidate
    against a specific JD using LLM.
    """
    resume_parsed = all_resumes.get(candidate_filename, {})
    jd_parsed     = all_jds.get(jd_filename, {})

    if not resume_parsed or not jd_parsed:
        print(f"Missing data for {candidate_filename}")
        return None

    # Build context for LLM
    summary_template = """
You are an expert Recruiter reviewing a candidate resume against a Job Description.

JOB DESCRIPTION:
{jd_json}

CANDIDATE RESUME:
{resume_json}

MATCH SCORES:
- Overall Match   : {final_score}
- Relevance Score : {relevance_score}
- Experience      : {resume_years} years (required: {required_years} years)

Based on the above, generate a structured evaluation in the following JSON format:
{
    "candidate_name"     : string,
    "overall_verdict"    : "Strong Match" | "Good Match" | "Partial Match" | "Weak Match",
    "match_summary"      : string (2-3 sentences overview of why this candidate fits or doesnt fit),
    "key_strengths"      : [string] (top 2-3 reasons this candidate is suitable),
    "skill_gaps"         : [string] (skills in JD that candidate is missing),
    "experience_summary" : string (brief summary of relevant experience),
    "recommendation"     : string (1-2 sentences final recommendation for the recruiter)
}

Rules:
1. Return ONLY valid JSON. No preamble, no explanation, no markdown blocks.
2. Be specific — reference actual skills, projects, companies from the resume.
3. Do not make anything up. Only use information present in the resume.
4. Keep each string concise and professional.
"""

    formatted_prompt = summary_template \
        .replace("{jd_json}",         json.dumps(jd_parsed, indent=2)) \
        .replace("{resume_json}",     json.dumps(resume_parsed, indent=2)) \
        .replace("{final_score}",     scores["percentage"]) \
        .replace("{relevance_score}", scores["relevance_score"]) \
        .replace("{resume_years}",    str(scores["resume_years"])) \
        .replace("{required_years}",  str(scores.get("required_years", "N/A")))

    response = llm.invoke(formatted_prompt).content

    # Parse response
    if "```json" in response:
        json_str = response.split("```json")[-1].split("```")[0].strip()
    elif "```" in response:
        json_str = response.split("```")[1].strip()
    elif "</think>" in response:
        json_str = response.split("</think>")[-1].strip()
    else:
        json_str = response.strip()

    json_str = clean_json_str(json_str)

    try:
        summary = json.loads(json_str)
        return summary
    except json.JSONDecodeError as e:
        print(f"Failed to parse summary for {candidate_filename}: {e}")
        return None


def generate_all_summaries(top_candidates, jd_filename,
                           all_resumes, all_jds, llm):
    """
    Generates summaries for all top candidates.
    top_candidates → output from get_top_candidates()
    """
    all_summaries = {}

    print(f"\n Generating summaries for top {len(top_candidates)} candidates...")
    print("=" * 65)

    for rank, (filename, scores) in enumerate(top_candidates, start=1):
        print(f"\n Generating summary for #{rank}: {filename}")

        summary = generate_candidate_summary(
            candidate_filename = filename,
            jd_filename        = jd_filename,
            all_resumes        = all_resumes,
            all_jds            = all_jds,
            llm                = llm,
            scores             = scores
        )

        if summary:
            all_summaries[filename] = {
                "rank"   : rank,
                "scores" : scores,
                "summary": summary
            }

            # Display summary
            print(f"\n{'='*65}")
            print(f"#{rank} — {summary.get('candidate_name', filename)}")
            print(f"{'='*65}")
            print(f"Verdict     : {summary.get('overall_verdict')}")
            print(f"Match Score : {scores['percentage']}")
            print(f"\nMatch Summary:")
            print(f"  {summary.get('match_summary')}")
            print(f"\nKey Strengths:")
            for s in (summary.get('key_strengths') or []):
                print(f"  {s}")
            print(f"\nSkill Gaps:")
            for s in (summary.get('skill_gaps') or []):
                print(f"   {s}")
            print(f"\nExperience Summary:")
            print(f"  {summary.get('experience_summary')}")
            print(f"\nRecommendation:")
            print(f"  {summary.get('recommendation')}")
        else:
            print(f" Skipped {filename} — summary generation failed")

    return all_summaries


In [18]:
# Load JSONs
with open(r"..\output_json_jd\all_jd.json", "r") as f:
    all_jds = json.load(f)

with open(r"..\output_jsons_resume\all_resumes.json", "r") as f:
    all_resumes = json.load(f)

# top_k = resume_collection._collection.count()
# Step 1- Get top Candidates using retrival of Vector databases.
# top_candidates = get_top_candidates(
#     jd_filename       = "PM_JD.txt",
#     resume_collection = resume_collection,
#     jd_collection     = jd_collection,
#     top_n             = 3,
#     top_k             = top_k
# )


# Step 2 — Generate summaries (Phase 6)
summaries = generate_all_summaries(
    top_candidates = cloud_top_candidates,
    jd_filename    = "DS_JD.txt",
    all_resumes    = all_resumes,
    all_jds        = all_jds,
    llm            = llm_groq
)


 Generating summaries for top 3 candidates...

 Generating summary for #1: Komal_Kamble_1 (1).pdf

#1 — Komal Kamble
Verdict     : Good Match
Match Score : 77.96%

Match Summary:
  Komal Kamble is a strong candidate for the Data Scientist position at Version 1, with 2.8 years of experience in data science and a solid foundation in Python, PySpark, and SQL. Her experience in developing predictive models and data storytelling skills align well with the job requirements. However, she lacks direct experience with AWS services and financial modeling.

Key Strengths:
  Developed end-to-end data science projects in production environments
  Applied statistical and machine learning techniques to solve business problems
  Strong foundation in Python, PySpark, and SQL

Skill Gaps:
   AWS services (SageMaker, Glue, Redshift, Lambda)
   Financial principles and quantitative analysis

Experience Summary:
  Komal has 2.8 years of experience in data science, with a focus on developing predictive mod

In [19]:
print(resume_collection._collection.count())

60


Add code below to add database in supabase.

In [21]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client


load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

if not SUPABASE_URL or not SUPABASE_SERVICE_ROLE_KEY:
    raise ValueError("Missing SUPABASE_URL or SUPABASE_SERVICE_ROLE_KEY in .env")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)


BASE_DIR = Path.cwd().parent

ALL_RESUMES_PATH = BASE_DIR / "output_jsons_resume" / "all_resumes.json"
ALL_JD_PATH = BASE_DIR / "output_json_jd" / "all_jd.json"
ALL_RESUME_CHUNKS_PATH = BASE_DIR / "chunks" / "all_resume_chunks.json"


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def store_resumes_in_supabase(all_resumes_path=ALL_RESUMES_PATH):
    all_resumes = load_json(all_resumes_path)

    rows = []
    for filename, resume_json in all_resumes.items():
        rows.append({
            "filename": filename,
            "resume_json": resume_json,
        })

    if not rows:
        print("No resumes found.")
        return

    response = (
        supabase.table("resumes")
        .upsert(rows, on_conflict="filename")
        .execute()
    )

    print(f"Stored/updated {len(rows)} resumes in Supabase.")
    return response


def store_jds_in_supabase(all_jd_path=ALL_JD_PATH):
    all_jds = load_json(all_jd_path)

    rows = []
    for filename, jd_json in all_jds.items():
        rows.append({
            "filename": filename,
            "jd_json": jd_json,
        })

    if not rows:
        print("No job descriptions found.")
        return

    response = (
        supabase.table("job_descriptions")
        .upsert(rows, on_conflict="filename")
        .execute()
    )

    print(f"Stored/updated {len(rows)} job descriptions in Supabase.")
    return response


def store_resume_chunks_in_supabase(all_resume_chunks_path=ALL_RESUME_CHUNKS_PATH):
    all_resume_chunks = load_json(all_resume_chunks_path)

    rows = []
    for resume_filename, chunks in all_resume_chunks.items():
        for chunk_index, chunk in enumerate(chunks):
            rows.append({
                "resume_filename": resume_filename,
                "chunk_index": chunk_index,
                "chunk_type": chunk.get("type"),
                "content": chunk.get("content", ""),
                "chunk_json": chunk,
            })

    if not rows:
        print("No resume chunks found.")
        return

    batch_size = 500

    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]

        supabase.table("resume_chunks").upsert(
            batch,
            on_conflict="resume_filename,chunk_index"
        ).execute()

    print(f"Stored/updated {len(rows)} resume chunks in Supabase.")


def sync_all_to_supabase():
    store_resumes_in_supabase()
    store_jds_in_supabase()
    store_resume_chunks_in_supabase()
    print("Supabase sync completed.")



In [22]:
sync_all_to_supabase()

Stored/updated 7 resumes in Supabase.
Stored/updated 3 job descriptions in Supabase.
Stored/updated 53 resume chunks in Supabase.
Supabase sync completed.


#### Evaluation 

##### Precision@k

In [17]:
# Precision@k - Retrieval metrics
def evaluate_precision_at_k(ground_truth, jd_filename,
                             resume_collection, jd_collection,
                             top_n=3, top_k=60):
    """
    Precision@K = number of relevant candidates in top K / K
    """
    relevant  = set(ground_truth.get(jd_filename, []))

    if not relevant:
        print(f"No ground truth defined for: {jd_filename}")
        return None

    top_candidates = get_top_candidates(
        jd_filename       = jd_filename,
        resume_collection = resume_collection,
        jd_collection     = jd_collection,
        top_n             = top_n,
        top_k             = top_k
    )

    retrieved  = [filename for filename, _ in top_candidates]
    hits       = [f for f in retrieved if f in relevant]
    precision  = round(len(hits) / len(retrieved), 4) if retrieved else 0.0

    print(f"\nPrecision@{top_n} for {jd_filename}")
    print("=" * 55)
    print(f"  Retrieved        : {retrieved}")
    print(f"  Relevant (truth) : {list(relevant)}")
    print(f"  Hits             : {hits}")
    print(f"  Precision@{top_n}    : {len(hits)}/{len(retrieved)} = {round(precision * 100, 1)}%")
    print(f"  Verdict          : {interpret_precision(precision, top_n)}")

    return {
        "jd_filename" : jd_filename,
        "precision"   : precision,
        "percentage"  : f"{round(precision * 100, 1)}%",
        "hits"        : hits,
        "retrieved"   : retrieved,
        "relevant"    : list(relevant)
    }


def interpret_precision(precision, k):
    if precision == 1.0:
        return f"Perfect — all {k} retrieved candidates are relevant"
    elif precision >= 0.67:
        return "Good — majority of retrieved candidates are relevant"
    elif precision >= 0.33:
        return "Acceptable — at least 1 relevant candidate retrieved"
    else:
        return "Poor — retrieved candidates are mostly irrelevant"


def evaluate_all_jds_precision(ground_truth, resume_collection,
                                jd_collection, top_n=3, top_k=53):
    """
    Run Precision@K across all JDs and report average.
    """
    results = {}

    for jd_filename in ground_truth.keys():
        result = evaluate_precision_at_k(
            ground_truth      = ground_truth,
            jd_filename       = jd_filename,
            resume_collection = resume_collection,
            jd_collection     = jd_collection,
            top_n             = top_n,
            top_k             = top_k
        )
        if result:
            results[jd_filename] = result

    avg_precision = round(
        sum(r["precision"] for r in results.values()) / len(results), 4
    ) if results else 0.0

    print(f"\nOverall Precision@{top_n} Summary")
    print("=" * 55)
    print(f"\n  {'JD':<25} {'Precision':>10}  {'Hits'}")
    print(f"  {'-'*50}")
    for jd, r in results.items():
        bar_len = int(r["precision"] * 20)
        bar     = "X" * bar_len + "." * (20 - bar_len)
        print(f"  {jd:<25} {r['percentage']:>10}  {bar}")

    print(f"\n  Average Precision@{top_n} : {round(avg_precision * 100, 1)}%")
    print(f"  Verdict              : {interpret_precision(avg_precision, top_n)}")

    return results, avg_precision


In [ ]:
# Step 1 — Define ground truth
# Read each JD and each resume yourself
# decide which resumes genuinely fit each JD
ground_truth = {
    "DS_JD.txt": ["Komal_Kamble_1 (1).pdf","MeetLad_Resume.pdf", "Jay Kumar Behera_CV_DS.pdf"],
        "PM_JD.txt": ["Abhishek_Shaurya_Resume.pdf"],
        "SWE_JD.txt": ["SOUMYADEEP_SEN_CV.pdf","MeetLad_Resume.pdf","MananDaxini-Updated-2026-t (1) (2).pdf","Prity-Kumari-Resume-DevOps-2025.pdf"]
}

# Step 3 — Run for ALL JDs at once
results, avg = evaluate_all_jds_precision(
    ground_truth      = ground_truth,
    resume_collection = resume_collection,
    jd_collection     = jd_collection,
    top_n             = 3,
    top_k             = 60
)

Fetched 5 chunks for JD: DS_JD.txt
BM25 index built with 60 resume chunks.
Total matches: 300 (5 JD chunks x top_k=60)
JD requires minimum 3.0 years experience

Top 3 Candidates for JD: DS_JD.txt
Score = 70% relevance + 20% role fit + 10% experience fit
Hybrid chunk score = 80% vector + 20% BM25

#1 - Komal_Kamble_1 (1).pdf
     Final Score     : 77.96%
     Relevance Score : 69.47%
     Role Fit        : 100.0%
     Experience Fit  : 93.33%
     Experience      : 2.8 yrs
     Matched Groups  : data_title, programming, ml, analytics, statistics, ai

     JD Requirement Breakdown (JD-Centric):
       JD Requirement       Resume Section         Vector     BM25   Hybrid
       ----------------------------------------------------------------------
       skills               skills                 0.7159   0.7171   0.7161
       qualifications       summary                0.6560   0.9195   0.7087
       summary              experience             0.6511   0.8344   0.6878
       responsibil

In [39]:
def evaluate_faithfulness(
    candidate_filename,
    jd_filename,
    all_resumes,
    all_jds,
    summary,
    llm
):


    resume_parsed = all_resumes.get(candidate_filename, {})
    jd_parsed     = all_jds.get(jd_filename, {})

    if not resume_parsed:
        print(f"❌ Resume not found: {candidate_filename}")
        return None

    eval_prompt = """
You are a strict fact-checker evaluating an AI-generated candidate summary.

Your job is to check whether every claim in the GENERATED SUMMARY is 
supported by the RESUME DATA below.

RESUME DATA (source of truth):
{resume_json}

JOB DESCRIPTION:
{jd_json}

GENERATED SUMMARY TO EVALUATE:
{summary_json}

For each field in the summary, check every claim against the resume data.

Return ONLY a JSON object in this exact format — no preamble, no markdown:
{{
    "faithfulness_score"  : float between 0.0 and 1.0,
    "total_claims"        : int,
    "supported_claims"    : int,
    "hallucinated_claims" : [
        {{
            "field"  : "which summary field this came from",
            "claim"  : "the exact claim that is not supported",
            "reason" : "why this is not supported by the resume"
        }}
    ],
    "verdict" : "Faithful" | "Partially Faithful" | "Not Faithful"
}}

Rules:
1. faithfulness_score = supported_claims / total_claims
2. A claim is supported if it can be directly verified from the resume data.
3. A claim is hallucinated if it adds information not present in the resume.
4. Paraphrasing is fine — only flag genuinely invented facts.
5. Skill gaps are valid if the skill is in the JD but absent from the resume.
6. Return ONLY valid JSON. No explanation outside the JSON.
"""

    formatted_prompt = eval_prompt \
        .replace("{resume_json}",  json.dumps(resume_parsed, indent=2)) \
        .replace("{jd_json}",      json.dumps(jd_parsed,     indent=2)) \
        .replace("{summary_json}", json.dumps(summary,        indent=2))

    response = llm.invoke(formatted_prompt).content

    # ── Parse response ────────────────────────────────────────
    if "```json" in response:
        json_str = response.split("```json")[-1].split("```")[0].strip()
    elif "```" in response:
        json_str = response.split("```")[1].strip()
    elif "</think>" in response:
        json_str = response.split("</think>")[-1].strip()
    else:
        json_str = response.strip()

    try:
        result = json.loads(json_str)
        return result
    except json.JSONDecodeError as e:
        print(f"❌ Failed to parse faithfulness eval: {e}")
        print(f"Raw response: {response[:300]}")
        return None


def run_faithfulness_eval(summaries, jd_filename,
                          all_resumes, all_jds, llm):
    """
    Runs faithfulness evaluation for all generated summaries.
    all_summaries → output from generate_all_summaries()
    """
    print("\nRunning Faithfulness Evaluation")
    print("=" * 60)

    eval_results = {}

    for filename, summary_data in summaries.items():
        summary = summary_data.get("summary", {})
        rank    = summary_data.get("rank", "?")

        print(f"\n⏳ Evaluating #{rank}: {filename}")

        result = evaluate_faithfulness(
            candidate_filename = filename,
            jd_filename        = jd_filename,
            all_resumes        = all_resumes,
            all_jds            = all_jds,
            summary            = summary,
            llm                = llm
        )

        if result:
            eval_results[filename] = result

            # ── Display result ────────────────────────────────
            score   = result.get("faithfulness_score", 0)
            verdict = result.get("verdict", "—")
            total   = result.get("total_claims",     0)
            supported = result.get("supported_claims", 0)
            hallucinations = result.get("hallucinated_claims", [])

            score_symbol = (
                "✅" if score >= 0.90 else
                "⚠️" if score >= 0.70 else
                "❌"
            )

            print(f"  {score_symbol} Verdict            : {verdict}")
            print(f"     Faithfulness Score : {round(score * 100, 1)}%")
            print(f"     Claims             : {supported}/{total} supported")

            if hallucinations:
                print(f"\n   Hallucinated Claims:")
                for h in hallucinations:
                    print(f"       Field  : {h.get('field')}")
                    print(f"       Claim  : {h.get('claim')}")
                    print(f"       Reason : {h.get('reason')}")
                    print()
        else:
            print(f"Evaluation failed for {filename}")

    # ── Overall summary ───────────────────────────────────────
    if eval_results:
        avg_score = sum(
            r.get("faithfulness_score", 0)
            for r in eval_results.values()
        ) / len(eval_results)

        print(f"\nOverall Faithfulness Score: {round(avg_score * 100, 1)}%")
        print("-" * 60)
        for filename, result in eval_results.items():
            score = result.get("faithfulness_score", 0)
            print(f"  {filename:<45} {round(score * 100, 1)}%")

    return eval_results

In [40]:
# ── Run ───────────────────────────────────────────────────────
# Assumes all_summaries is already generated from Phase 6
faithfulness_results = run_faithfulness_eval(
    summaries = summaries,
    jd_filename   = "PM_JD.txt",
    all_resumes   = all_resumes,
    all_jds       = all_jds,
    llm           = llm_groq
)


Running Faithfulness Evaluation

⏳ Evaluating #1: Abhishek_Shaurya_Resume.pdf
  ⚠️ Verdict            : Partially Faithful
     Faithfulness Score : 80.0%
     Claims             : 10/13 supported

   Hallucinated Claims:
       Field  : match_summary
       Claim  : However, he lacks direct experience with AI-powered solutions and no-code platforms.
       Reason : The resume mentions experience with AI audit co-pilot, but not no-code platforms.

       Field  : key_strengths
       Claim  : Ability to drive platform modernization and reduce engineering hours
       Reason : The resume mentions reducing engineering hours, but not platform modernization.

       Field  : recommendation
       Claim  : Abhishek is a strong candidate for the Full Stack AI Product Manager role, but requires training or experience in AI-powered solutions and no-code platforms to meet the job requirements.
       Reason : The resume mentions experience with AI audit co-pilot, but not no-code platforms.

  

#### Retrival Relevance - LLM as a Judge

In [2]:
def evaluate_retrieval_relevance(jd_filename, top_candidates,
                                  all_jds, all_resumes, llm):
    """
    Uses LLM as judge to score how relevant each retrieved
    resume is to the JD on a scale of 1-5.

    Relevance Scale:
    5 - Perfect Match   - candidate clearly fits the role
    4 - Good Match      - candidate fits most requirements
    3 - Partial Match   - candidate fits some requirements
    2 - Weak Match      - candidate has few relevant skills
    1 - Not Relevant    - completely wrong profile

    Accepted thresholds (industry standard):
    >= 4.0  - Excellent retrieval
    >= 3.0  - Good retrieval
    >= 2.0  - Acceptable retrieval
    <  2.0  - Poor retrieval - needs improvement
    """

    jd_parsed  = all_jds.get(jd_filename, {})
    compact_jd = {
        "job_title"          : jd_parsed.get("job_title"),
        "skills_required"    : jd_parsed.get("skills_required"),
        "experience_required": jd_parsed.get("experience_required"),
        "job_summary"        : jd_parsed.get("job_summary")
    }

    relevance_results = {}
    print(f"\nEvaluating Retrieval Relevance for: {jd_filename}")
    print("=" * 60)

    for filename, scores in top_candidates:
        resume_parsed  = all_resumes.get(filename, {})
        compact_resume = {
            "name"      : resume_parsed.get("name"),
            "summary"   : resume_parsed.get("summary"),
            "skills"    : (resume_parsed.get("skills") or [])[:10],
            "experience": [
                {
                    "title"  : e.get("title"),
                    "company": e.get("company")
                }
                for e in (resume_parsed.get("experience") or [])[:2]
            ]
        }

        eval_template = """
You are an expert Technical Recruiter.
Rate how relevant this candidate is for the job on a scale of 1 to 5.

JD: {jd_json}
CANDIDATE: {resume_json}

Return ONLY valid JSON:
{
    "relevance_score" : integer (1 to 5),
    "relevance_label" : "Perfect Match" | "Good Match" | "Partial Match" | "Weak Match" | "Not Relevant",
    "reason"          : string (1 sentence only)
}

Scoring Guide:
5 - Perfect Match   - candidate clearly fits the role
4 - Good Match      - candidate fits most requirements
3 - Partial Match   - candidate fits some requirements
2 - Weak Match      - candidate has few relevant skills
1 - Not Relevant    - completely wrong profile

Rules:
1. Return ONLY valid JSON. No preamble, no markdown.
2. Base score only on JD requirements vs candidate profile.
3. Do not make anything up.
4. While writing the reason, make sure to be smart about it.
"""

        formatted_prompt = eval_template \
            .replace("{jd_json}",     json.dumps(compact_jd,     indent=2)) \
            .replace("{resume_json}", json.dumps(compact_resume, indent=2))

        response = llm.invoke(formatted_prompt).content

        if "```json" in response:
            json_str = response.split("```json")[-1].split("```")[0].strip()
        elif "```" in response:
            json_str = response.split("```")[1].strip()
        elif "</think>" in response:
            json_str = response.split("</think>")[-1].strip()
        else:
            json_str = response.strip()

        json_str = re.sub(r",\s*([}\]])", r"\1", json_str)

        try:
            result = json.loads(json_str)
            relevance_results[filename] = {
                "candidate_name" : resume_parsed.get("name", filename),
                "system_score"   : scores["percentage"],
                "relevance_score": result.get("relevance_score"),
                "relevance_label": result.get("relevance_label"),
                "reason"         : result.get("reason")
            }

            print(f"\n{resume_parsed.get('name', filename)}")
            print(f"  System Score     : {scores['percentage']}")
            print(f"  Relevance Score  : {result.get('relevance_score')}/5 - {result.get('relevance_label')}")
            print(f"  Reason           : {result.get('reason')}")

        except json.JSONDecodeError:
            print(f"Failed to parse relevance for {filename}")

    # Summary
    score_list    = [
        v["relevance_score"]
        for v in relevance_results.values()
        if v.get("relevance_score")
    ]
    avg_relevance = round(sum(score_list) / len(score_list), 2) if score_list else 0.0

    # Correlation check — system score vs LLM relevance
    print(f"\nRetrieval Relevance Summary")
    print("=" * 60)
    print(f"\n{'Candidate':<30} {'System Score':>12} {'LLM Score':>10} {'Label':<20}")
    print(f"{'-'*75}")
    for filename, r in relevance_results.items():
        print(f"  {r['candidate_name']:<28} "
              f"{r['system_score']:>12} "
              f"{str(r['relevance_score'])+'/5':>10} "
              f"{r['relevance_label']:<20}")

    print(f"\nAverage Relevance Score : {avg_relevance}/5")
    print(f"Threshold Interpretation: {interpret_relevance(avg_relevance)}")

    # Correlation warning
    check_score_correlation(relevance_results)

    return relevance_results, avg_relevance


def interpret_relevance(avg_score):
    """
    Industry standard thresholds for retrieval relevance.
    Source: RAGAS framework benchmarks 2025
    >= 4.0 (0.8 normalised) - Excellent
    >= 3.0 (0.6 normalised) - Good
    >= 2.0 (0.4 normalised) - Acceptable
    <  2.0                  - Poor
    """
    if avg_score >= 4.0:
        return "Excellent - system surfacing highly relevant candidates"
    elif avg_score >= 3.0:
        return "Good - most candidates are relevant"
    elif avg_score >= 2.0:
        return "Acceptable - some irrelevant candidates appearing"
    else:
        return "Poor - retrieval needs improvement"


def check_score_correlation(relevance_results):
    """
    Checks if system ranking correlates with LLM relevance scores.
    If high system score = high relevance score -> pipeline is working.
    If they dont correlate -> scoring logic needs review.
    """
    print(f"\nCorrelation Check (System Score vs LLM Relevance):")
    print("-" * 50)

    mismatches = []
    for filename, r in relevance_results.items():
        system_pct     = float(r["system_score"].replace("%", ""))
        relevance_score = r.get("relevance_score", 0)

        # Flag if high system score but low relevance or vice versa
        if system_pct >= 70 and relevance_score <= 2:
            mismatches.append(
                f"  WARNING: {r['candidate_name']} - "
                f"high system score ({r['system_score']}) "
                f"but low LLM relevance ({relevance_score}/5)"
            )
        elif system_pct <= 55 and relevance_score >= 4:
            mismatches.append(
                f"  WARNING: {r['candidate_name']} - "
                f"low system score ({r['system_score']}) "
                f"but high LLM relevance ({relevance_score}/5)"
            )

    if mismatches:
        print("Mismatches found - scoring logic may need review:")
        for m in mismatches:
            print(m)
    else:
        print("No major mismatches - system scores align with LLM relevance")

In [3]:
def evaluate_relevance_for_jd(jd_filename, resume_collection, jd_collection,
                              all_jds, all_resumes, llm,
                              top_n=3, top_k=60):
    top_candidates = get_top_candidates(
        jd_filename=jd_filename,
        resume_collection=resume_collection,
        jd_collection=jd_collection,
        top_n=top_n,
        top_k=top_k
    )

    return evaluate_retrieval_relevance(
        jd_filename=jd_filename,
        top_candidates=top_candidates,
        all_jds=all_jds,
        all_resumes=all_resumes,
        llm=llm
    )

In [19]:
relevance_results, avg = evaluate_relevance_for_jd(
    jd_filename="DS_JD.txt",
    resume_collection=resume_collection,
    jd_collection=jd_collection,
    all_jds=all_jds,
    all_resumes=all_resumes,
    llm=llm_gemini,
    top_n=3,
    top_k=60
)

Fetched 5 chunks for JD: DS_JD.txt
BM25 index built with 60 resume chunks.
Total matches: 300 (5 JD chunks x top_k=60)
JD requires minimum 3.0 years experience

Top 3 Candidates for JD: DS_JD.txt
Score = 70% relevance + 20% role fit + 10% experience fit
Hybrid chunk score = 80% vector + 20% BM25

#1 - Komal_Kamble_1 (1).pdf
     Final Score     : 77.96%
     Relevance Score : 69.47%
     Role Fit        : 100.0%
     Experience Fit  : 93.33%
     Experience      : 2.8 yrs
     Matched Groups  : data_title, programming, ml, analytics, statistics, ai

     JD Requirement Breakdown (JD-Centric):
       JD Requirement       Resume Section         Vector     BM25   Hybrid
       ----------------------------------------------------------------------
       skills               skills                 0.7159   0.7171   0.7161
       qualifications       summary                0.6560   0.9195   0.7087
       summary              experience             0.6511   0.8344   0.6878
       responsibil

##### Hallucination Check

In [27]:
def check_summary_hallucination(candidate_filename, jd_filename,
                                generated_summary,
                                all_resumes, all_jds, llm):
    resume_parsed = all_resumes.get(candidate_filename, {})
    jd_parsed = all_jds.get(jd_filename, {})

    if not resume_parsed or not jd_parsed:
        print("Missing resume or JD data.")
        return None

    hallucination_prompt = """
You are a careful hallucination evaluator for a resume screening system.

Your task:
Check whether the GENERATED SUMMARY makes any positive claim about the candidate
that is NOT supported by the RESUME JSON.

Important rules:
1. A claim is supported if it is directly stated OR reasonably paraphrased from the resume.
2. Do NOT require exact wording.
3. Do NOT mark skill gaps as hallucinations. Skill gaps describe what the candidate is missing.
4. Do NOT mark reasonable summaries as hallucinations.
5. Only mark a hallucination if the summary says the candidate HAS something that is absent from the resume.
6. JD requirements alone are not evidence that the candidate has that skill.
7. For every hallucinated claim, explain why it is unsupported.
8. For every supported claim, cite the resume evidence briefly.

Examples:
Resume: "Owns roadmap, backlog, and delivery"
Summary: "Has product ownership and roadmap management experience"
Decision: supported, not hallucination.

Resume: "Python, SQL, AWS"
Summary: "Strong technical skills in Python, SQL, and AWS"
Decision: supported, not hallucination.

Resume: "Led AWS cost optimization saving $38K"
Summary: "Led initiatives that resulted in cost savings"
Decision: supported, not hallucination.

Resume does not mention SageMaker.
Summary: "Has SageMaker experience"
Decision: hallucination.

JOB DESCRIPTION JSON:
{jd_json}

RESUME JSON:
{resume_json}

GENERATED SUMMARY JSON:
{summary_json}

Return ONLY valid JSON:
{
    "hallucination_present": true or false,
    "supported_claims": [
        {
            "claim": string,
            "resume_evidence": string
        }
    ],
    "hallucinated_claims": [
        {
            "claim": string,
            "why_unsupported": string
        }
    ],
    "hallucination_rate": number between 0 and 1,
    "verdict": "Pass" or "Fail",
    "reason": string
}

Hallucination rate formula:
number of hallucinated positive claims / total positive claims checked.
"""

    formatted_prompt = hallucination_prompt \
        .replace("{jd_json}", json.dumps(jd_parsed, indent=2)) \
        .replace("{resume_json}", json.dumps(resume_parsed, indent=2)) \
        .replace("{summary_json}", json.dumps(generated_summary, indent=2))

    response = llm.invoke(formatted_prompt).content

    if "```json" in response:
        json_str = response.split("```json")[-1].split("```")[0].strip()
    elif "```" in response:
        json_str = response.split("```")[1].strip()
    elif "</think>" in response:
        json_str = response.split("</think>")[-1].strip()
    else:
        json_str = response.strip()

    json_str = re.sub(r",\s*([}\]])", r"\1", json_str)

    try:
        result = json.loads(json_str)

        print(f"\nHallucination Check: {candidate_filename}")
        print("=" * 60)
        print(f"Hallucination Present : {result.get('hallucination_present')}")
        print(f"Hallucination Rate    : {result.get('hallucination_rate')}")
        print(f"Verdict               : {result.get('verdict')}")
        print(f"Reason                : {result.get('reason')}")

        hallucinated_claims = result.get("hallucinated_claims") or []
        if hallucinated_claims:
            print("\nHallucinated Claims:")
            for item in hallucinated_claims:
                print(f"- {item.get('claim')}")
                print(f"  Why: {item.get('why_unsupported')}")

        return result

    except json.JSONDecodeError:
        print("Failed to parse hallucination check response.")
        print(response)
        return None

In [28]:
hallucination_results = {}

for filename, data in summaries.items():
    hallucination_results[filename] = check_summary_hallucination(
        candidate_filename=filename,
        jd_filename="PM_JD.txt",
        generated_summary=data["summary"],
        all_resumes=all_resumes,
        all_jds=all_jds,
        llm=llm_groq
    )


Hallucination Check: Abhishek_Shaurya_Resume.pdf
Hallucination Present : False
Hallucination Rate    : 0
Verdict               : Pass
Reason                : No hallucinations found in the generated summary.

Hallucinated Claims:
- Hands-on experience with AI-powered solutions using no-code platforms (Replit, Lovable, Cursor)
  Why: No direct experience with AI-powered solutions and no-code platforms mentioned in the resume.
- Understanding of UX & Design systems
  Why: No direct experience or skills related to UX & Design systems mentioned in the resume.

Hallucination Check: MeetLad_Resume.pdf
Hallucination Present : False
Hallucination Rate    : 0
Verdict               : Pass
Reason                : Candidate has relevant experience and skills, but lacks direct experience in AI product management and full-stack building.

Hallucinated Claims:
- Hands-on experience with design first products
  Why: Not mentioned in the resume, and JD requires hands-on experience with design first pr